In [ ]:
# 연관규칙 실습용 필요한 패키지 설치
from mlxtend.preprocessing import TransactionEncoder
# 리스트 형태의 거래 데이터를 원-핫 인코딩 형태의 DataFrame으로 변환하는 클래스 호출
# 연관 분석을 위한 입력 형태(0/1 매트릭스)로 전처리하는 기능 제공

from mlxtend.frequent_patterns import apriori
# Apriori 알고리즘을 사용하여 빈발 항목집합(frequent itemsets)을 추출하는 함수 호출
# 최소 지지도(min_support) 기준을 만족하는 아이템 조합을 탐색하여 반환함

from mlxtend.frequent_patterns import association_rules
# 빈발 항목집합을 기반으로 연관 규칙(Association Rules)을 도출하는 함수 호출
# 지지도(support), 신뢰도(confidence), 향상도(lift) 등을 계산하여 연관성 있는 규칙 도출 수행

import pandas as pd
# 표 형식의 데이터 처리 및 분석 기능을 제공하는 pandas 라이브러리 임포트
# 주로 DataFrame 구조를 통해 거래 데이터 및 규칙 결과를 저장하고 조작함

import numpy as np
# 수치 연산 및 배열 처리를 위한 numpy 라이브러리 임포트
# 이산형 행렬 변환, 조건 필터링, 지표 계산 등 연관 규칙 분석 과정에서 필수적인 수학 연산 지원

In [2]:
# 연관규칙 실습용 데이터 불러오기
# https://www.kaggle.com/datasets/sewonghwang/market-basket
df = pd.read_csv("datasets/market_basket.csv")

# 데이터 샘플 확인
df.head()

,cust_cd,std_dt,prdct_cd,prdct_nm
0,C617077280704,2021-06-19,A10001,tropical fruit
1,C617077280704,2021-06-19,A10002,whole milk
2,C617077280704,2021-06-19,A10003,pip fruit
3,C617077280704,2021-06-19,A10004,other vegetables
4,C617077280704,2021-06-19,A10005,cream


In [ ]:
# apriori 모델 적용을 위한 품목 리스트 가공

itemset = df.drop_duplicates(
    ['cust_cd', 'std_dt', 'prdct_nm']).groupby(
    ['cust_cd','std_dt'])['prdct_nm'].apply(list)
# 동일 고객(cust_cd)이 같은 날(std_dt) 중복 구매한 상품(prdct_nm)을 제거하기 위해 세 컬럼 기준으로 중복 행 제거 수행
# 고객-날짜 단위로 groupby()를 수행하여, 해당 거래에서 구매한 상품명을 리스트 형식으로 묶어 반환함
# 즉, 하나의 거래(고객의 하루 구매)를 하나의 품목 리스트로 구성하는 작업 수행
# apriori 분석에서 요구하는 거래 단위의 itemset(품목 목록) 형태로 데이터를 구조화함

itemset = pd.DataFrame(itemset).reset_index().drop(
    ['cust_cd', 'std_dt'], axis='columns')
# 위에서 생성된 시리즈(itemset)를 데이터프레임으로 변환하고, 인덱스로 있던 'cust_cd', 'std_dt'를 일반 열로 복원한 후 제거함
# 이제 각 행은 하나의 거래(하루치 구매 목록)를 나타내며, 해당 행에는 상품명 리스트만 남도록 정제함

itemset = itemset.squeeze()
# 단일 열 구조의 데이터프레임을 pandas 시리즈로 변환함
# 각 거래가 리스트 형태로 저장된 시리즈가 되어 TransactionEncoder에 입력 가능한 구조로 가공됨

itemset.head()
# 최종 생성된 거래-품목 리스트 구조의 상위 5개를 출력하여 가공이 정상적으로 수행되었는지 확인함
# 출력값은 다음 형태와 유사함:  
# 0    [우유, 계란, 빵]  
# 1    [콜라, 치킨, 맥주]  
# → 각 행이 개별 거래(transaction)를 나타내는 리스트형 데이터로 구성됨


0    [beef, herbs, tropical fruit, whole milk, chic...
1    [sugar, packaged fruit/vegetables, sausage, sp...
2    [berries, tropical fruit, fruit/vegetable juic...
3    [yogurt, beef, cream, herbs, chicken, bottled ...
4    [berries, beef, yogurt, specialty bar, bottled...
Name: prdct_nm, dtype: object

In [ ]:
# apriori 모델 적용을 위한 장바구니 - 품목 더미 가공

encoder = TransactionEncoder()
# TransactionEncoder 객체를 생성함
# 리스트 형식으로 구성된 거래 데이터(itemset)를 원-핫 인코딩 형태로 변환하는 도구임
# 각 거래를 품목 기준으로 0 또는 1 값으로 표현하기 위한 전처리 준비 단계임

encoder_T = encoder.fit(itemset).transform(itemset)
# TransactionEncoder 객체에 거래 데이터(itemset)를 입력하여 학습(fit)과 변환(transform)을 동시에 수행함
# 학습(fit)은 전체 품목 목록을 파악하고, 변환(transform)은 각 거래에 대해 품목 존재 여부를 0 또는 1로 표시하는 이진 행렬 생성함
# 반환 결과는 각 거래(행) × 품목(열) 구조의 NumPy 배열 형태임

# 데이터프레임으로 변경
itemset_matrix = pd.DataFrame(encoder_T, columns=encoder.columns_)
# 원-핫 인코딩 결과(NumPy 배열)를 pandas DataFrame으로 변환함
# 열 이름(columns)은 encoder.columns_ 속성을 사용하여 각 품목명을 할당함
# 결과적으로 거래-품목 이진 매트릭스가 생성되며, apriori 알고리즘 입력 형식에 적합한 구조로 변환됨

itemset_matrix.head()
# 가공된 거래-품목 이진 행렬의 상위 5개 행을 출력하여 구조가 올바르게 생성되었는지 확인함
# 출력 예:  
# | milk | bread | beer | ... |  
# |  1   |   0   |  1   | ... |  
# → 각 행은 거래, 각 열은 품목이며, 값은 해당 품목의 구매 여부(0 또는 1)를 나타냄


,beef,berries,beverages,bottled beer,bottled water,brown bread,butter,butter milk,canned beer,chicken,...,sparkling wine,specialty bar,specialty chocolate,sugar,syrup,tropical fruit,turkey,white wine,whole milk,yogurt
0,True,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,True,False,False,True,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,True,True,False,True,False,False,False,False
2,False,True,False,False,False,False,False,False,False,True,...,False,False,False,False,False,True,False,False,False,False
3,True,False,False,False,True,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,True
4,True,True,False,False,True,False,False,False,False,True,...,False,True,False,False,False,True,False,False,False,True


In [ ]:
# 지지도 0.1 기준으로 apriori 적용

itemset_apriori = apriori(itemset_matrix, min_support=0.01, use_colnames=True)
# mlxtend의 apriori 함수를 사용하여 빈발 항목집합(frequent itemsets)을 도출함
# itemset_matrix는 거래(행) × 품목(열)의 0/1 매트릭스 구조이며, 각 셀은 품목의 포함 여부를 의미함
# min_support=0.01은 최소 지지도 기준으로 전체 거래의 1% 이상에서 등장한 아이템 조합만 추출함
# use_colnames=True는 결과에서 품목명을 정수 인덱스가 아닌 실제 이름으로 표시함

itemset_apriori.head()
# apriori 결과의 상위 5개 항목집합을 출력함
# 결과는 다음과 같은 열을 포함함:
# - `support`: 해당 항목집합의 지지도 (빈도 / 전체 거래 수)
# - `itemsets`: 추출된 빈발 항목집합 (frozenset 자료형으로 저장됨)
# 예시:
# | support | itemsets        |
# |---------|-----------------|
# |  0.15   | {milk}          |
# |  0.12   | {bread}         |
# |  0.10   | {milk, bread}   |


,support,itemsets
0,0.166612,(beef)
1,0.105074,(berries)
2,0.017010,(beverages)
3,0.025754,(bottled beer)
4,0.095191,(bottled water)


| 용어                | 설명                                                              |
| ----------------- | --------------------------------------------------------------- |
| **지지도 (support)** | 전체 거래 중 해당 항목집합이 등장한 비율<br>`support = (항목 포함 거래 수) / (전체 거래 수)` |
| **빈발 항목집합**       | 주어진 지지도 기준(min\_support) 이상으로 등장하는 아이템 조합                       |
| **frozenset**     | 변경 불가능한 집합 자료형으로, itemsets 컬럼에 저장됨                              |


In [ ]:
# 향상도 5 이상 상품 조합 추출

# **향상도(Lift)**란?
# 정의
# **향상도(Lift)**는 어떤 상품 A를 샀을 때 상품 B도 같이 살 가능성이, B를 무작위로 살 가능성보다 몇 배 높은지를 나타내는 지표.

association_rules(itemset_apriori, metric="lift", min_threshold=5)
# apriori 결과로 도출된 빈발 항목집합(itemset_apriori)을 기반으로 연관 규칙을 생성하는 함수 호출
# metric="lift"는 규칙의 평가 지표로 **향상도(lift)**를 사용함
# lift는 두 상품 간의 **실제 동시 발생 빈도**가 **각각의 발생 확률의 곱**보다 얼마나 더 높은지를 나타냄
# min_threshold=5는 향상도가 5 이상인 규칙만 필터링함
# 즉, A → B라는 규칙이 있을 때 B가 A와 동시에 등장할 확률이 B가 독립적으로 등장할 확률보다 5배 이상 높을 경우만 추출함
# 반환값은 antecedents (조건), consequents (결과), support, confidence, lift 등을 포함한 연관 규칙의 DataFrame 구조임


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction
0,"(beef, ham)",(bottled water),0.028179,0.095191,0.014585,0.517601,5.437508,0.011903,1.875645
1,(bottled water),"(beef, ham)",0.095191,0.028179,0.014585,0.153223,5.437508,0.011903,1.147670
2,"(beef, yogurt)",(bottled water),0.023844,0.095191,0.011977,0.502311,5.276885,0.009707,1.818022
3,(bottled water),"(beef, yogurt)",0.095191,0.023844,0.011977,0.125820,5.276885,0.009707,1.116654
4,"(pastry, beef, cream)",(bottled water),0.030163,0.095191,0.014879,0.493301,5.182229,0.012008,1.785693
5,"(pastry, beef)","(bottled water, cream)",0.046916,0.047797,0.014879,0.317150,6.635276,0.012637,1.394453
6,"(pastry, bottled water)","(beef, cream)",0.037474,0.069878,0.014879,0.397059,5.682200,0.012261,1.542642
7,"(beef, cream)","(pastry, bottled water)",0.069878,0.037474,0.014879,0.212934,5.682200,0.012261,1.222929
8,"(bottled water, cream)","(pastry, beef)",0.047797,0.046916,0.014879,0.311299,6.635276,0.012637,1.383887
9,(bottled water),"(pastry, beef, cream)",0.095191,0.030163,0.014879,0.156310,5.182229,0.012008,1.149519


| 향상도(lift) 값 | 해석                                 |
| ----------- | ---------------------------------- |
| = 1         | A와 B는 서로 독립 (무관한 상품)               |
| > 1         | A를 샀을 때 B도 살 가능성이 일반보다 큼 (양의 상관관계) |
| < 1         | A를 샀을 때 B는 오히려 덜 사는 경향 (음의 상관관계)   |
| > **3\~5**  | **매우 강한 연관성**으로 간주함 (실무 기준)        |


| 열 이름 | 의미 | 해석 포인트 |
| --- | --- | --- |
| **antecedents** | 조건 아이템 집합 (If 부분) | A가 포함된 장바구니 기준 |
| **consequents** | 결과 아이템 집합 (Then 부분) | B가 조건 A 뒤에 등장한 규칙 |
| **antecedent support** | A가 등장한 비율 | 전체 거래 중 A가 포함된 비율 |
| **consequent support** | B가 등장한 비율 | 전체 거래 중 B가 포함된 비율 |
| **support** | A와 B가 함께 등장한 비율 | 전체 거래 중 A ∩ B의 비율 |
| **confidence** | A가 있을 때 B도 포함될 확률 | A → B의 조건부 확률 |
| **lift** | A가 있을 때 B가 나올 상대적 강화도 | 일반 확률 대비 A → B가 얼마나 강화되는지 |
| **leverage** | 기대되는 동시발생률과 실제 동시발생률의 차이 | 값이 클수록 의미 있는 연관성 |
| **conviction** | B가 A 없이 등장할 확률의 역수 | confidence가 1에 가까울수록 conviction은 무한대로 커짐 |

---

## ✅ 예시 행 해석 (0번 행 기준)

| 항목 | 값 | 해석 |
| --- | --- | --- |
| antecedents | (beef, ham) | "소고기와 햄을 함께 산 고객"이 |
| consequents | (bottled water) | **생수를 살 가능성**을 나타냄 |
| antecedent support | 0.0282 | 전체 거래 중 2.8%가 beef+ham 포함 |
| consequent support | 0.0952 | 전체 거래 중 9.5%가 bottled water 포함 |
| support | 0.0146 | beef+ham과 bottled water가 함께 등장한 비율 |
| confidence | 0.5176 | beef+ham을 산 고객 중 51.8%가 생수도 구매함 |
| lift | 5.4375 | 생수를 우연히 살 확률보다 5.4배 높음 → 강한 연관성 |
| leverage | 0.0119 | 실제 동시 구매율이 기대치보다 1.19%p 더 높음 |
| conviction | 1.8756 | 생수 없이 beef+ham만 나올 확률의 역수 → 높을수록 강한 규칙 |

---

## ✅ 실무적으로 유의미한 규칙 고르기 위한 기준

| 조건 | 기준값 |
| --- | --- |
| support | 0.01 이상 (너무 희귀한 조합 제거) |
| confidence | 0.5 이상 |
| lift | **3~5 이상이면 강한 규칙**으로 간주 |
| leverage | 0.01 이상이면 실질적인 의미 존재 |
| conviction | 1.2 이상이면 양의 방향성 존재 가능성 |

In [7]:
#################### 협업 필터링 #####################

In [ ]:
# 협업 필터링 실습용 필요한 패키지 설치

from sklearn.metrics.pairwise import cosine_similarity
# 사용자 또는 아이템 간의 유사도를 계산하는 함수 호출
# 벡터 간의 코사인 유사도를 계산하여 두 벡터 간 방향의 유사도를 수치화함
# 주로 사용자 기반(user-based) 또는 아이템 기반(item-based) 협업 필터링에서 선호 패턴의 유사도를 측정하기 위해 사용됨

from sklearn.metrics import mean_squared_error
# 예측된 평점과 실제 평점 간의 차이를 제곱하여 평균낸 평균 제곱 오차(MSE)를 계산하는 함수 호출
# 협업 필터링 모델의 예측 성능을 수치적으로 평가하기 위한 지표로 활용됨

import pandas as pd
# 데이터프레임 기반의 데이터 처리, 전처리, 피벗 테이블 생성, 평점 행렬 구성 등에 사용되는 필수 라이브러리 호출

import numpy as np
# 수치 연산, 배열 처리, 결측값 보간 및 벡터 연산 등을 위한 필수 수학 라이브러리 호출
# 평점 행렬 생성 및 유사도 계산 시 다양한 수학적 기능 제공


In [2]:
# 협업 필터링 실습용 데이터 불러오기
# https://www.kaggle.com/datasets/ayushimishra2809/movielens-dataset
df_movies = pd.read_csv("datasets/movies.csv")
df_ratings = pd.read_csv("datasets/ratings.csv")

# 데이터 샘플 확인
df_ratings.head()

,userId,movieId,rating,timestamp
0,1,16,4.0,1217897793
1,1,24,1.5,1217895807
2,1,32,4.0,1217896246
3,1,47,4.0,1217896556
4,1,50,4.0,1217896523


In [ ]:
# 고객, 영화 유사도 측정을 위한 전치 데이터셋 생성

# ratings 데이터와 movies 데이터 결합
df_merge = pd.merge(df_ratings, df_movies, on="movieId")
# ratings.csv와 movies.csv를 movieId 기준으로 병합함
# 각 평점 데이터에 영화 제목(title) 및 장르(genres) 정보를 추가하여 해석 가능한 형태로 확장함
# 이후 평점 행렬의 열을 제목(title) 기준으로 구성하기 위한 사전 처리 작업임

# 고객-아이템 평점 행렬 생성
df_merge_pivot = df_merge.pivot_table("rating", "userId", "title")
# 사용자(userId)를 행(index), 영화 제목(title)을 열(columns), 평점(rating)을 값(value)으로 구성한 피벗 테이블 생성
# 협업 필터링에서 사용자 간 또는 아이템 간 유사도 측정의 입력으로 사용될 평점 행렬 생성 과정임
# 각 셀은 해당 사용자가 해당 영화를 평가한 평점을 의미함

# 결측 0으로 변환
df_merge_pivot_null = df_merge_pivot.fillna(0)
# 사용자가 평가하지 않은 영화는 NaN으로 표시되어 있음
# 유사도 계산을 위해 모든 결측값을 0으로 대체함
# 평점이 0이라는 것은 '관측되지 않은 항목'을 의미하며, 계산 편의를 위해 0으로 간주함

# 아이템-사용자 평점 행렬로 전치
df_merge_pivot_T = df_merge_pivot_null.T
# 영화 제목(title)을 행(index), 사용자(userId)를 열(columns)로 바꾸어 전치(Transpose)함
# 각 행은 하나의 영화이고, 각 열은 사용자별 평점 정보를 포함함
# 아이템 기반 협업 필터링(Item-based CF)에서 **영화 간 유사도 측정**을 위해 사용하는 구조임

df_merge_pivot_T.head()
# 전치된 평점 행렬의 상위 5개 행을 출력하여 구조를 확인함
# 각 행은 영화 제목, 각 열은 사용자 ID이며, 셀 값은 평점(또는 0)으로 구성됨


userId,1,2,3,4,5,6,7,8,9,10,...,659,660,661,662,663,664,665,666,667,668
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Hellboy': The Seeds of Creation (2004),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Round Midnight (1986),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.5
'Til There Was You (1997),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"'burbs, The (1989)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# 영화 유사도 행렬 생성
item_sim = cosine_similarity(df_merge_pivot_T)
# 전치된 영화-사용자 평점 행렬(df_merge_pivot_T)을 입력으로 하여 코사인 유사도를 계산함
# 각 영화(행)는 사용자 평점 벡터로 표현되며, cosine_similarity는 두 벡터 간 방향 유사도를 수치화함
# 결과는 영화 간 유사도를 나타내는 2차원 배열로 반환되며, 대칭 행렬의 형태를 가짐
# 값 범위는 [0, 1]로, 1에 가까울수록 두 영화의 평점 패턴이 유사함을 의미함

# 데이터 프레임 형태 변환
item_sim_df = pd.DataFrame(item_sim, index=df_merge_pivot_T.index,
                           columns=df_merge_pivot_T.index)
# NumPy 배열로 반환된 유사도 행렬을 pandas DataFrame으로 변환함
# 행과 열의 이름(index와 columns)을 영화 제목(title)으로 지정하여 해석 가능한 형태로 구성함
# item_sim_df는 영화 간의 유사도 정보를 담은 정방행렬이며, 특정 영화와 다른 영화들 간 유사도를 바로 조회할 수 있음

item_sim_df.head()
# 생성된 영화 간 유사도 행렬의 상위 5개 행을 출력함
# 각 행과 열은 영화 제목이며, 셀 값은 해당 두 영화 간의 코사인 유사도를 의미함


title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Til There Was You (1997),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...And Justice for All (1979),10 (1979),...,[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),a/k/a Tommy Chong (2005),eXistenZ (1999),loudQUIETloud: A Film About the Pixies (2006),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),1.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.342682,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.158272,0.0,0.098324,0.0
'Hellboy': The Seeds of Creation (2004),0.0,1.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.147486,0.0
'Round Midnight (1986),0.0,0.0,1.0,0.0,0.0,0.0,0.081094,0.000000,0.257012,0.680414,...,0.000000,0.227429,0.141421,0.0,0.100219,0.0,0.221581,0.0,0.098324,1.0
'Til There Was You (1997),0.0,0.0,0.0,1.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0
"'burbs, The (1989)",0.0,0.0,0.0,0.0,1.0,0.0,0.031610,0.231897,0.100923,0.000000,...,0.057358,0.000000,0.000000,0.0,0.212684,0.0,0.104192,0.0,0.161820,0.0


- **대각선은 항상 1.000** (자기 자신과의 유사도)
- 행 단위로 정렬하면 해당 영화와 유사한 영화를 추천할 수 있음

In [ ]:
# 500일의 썸머와 유사도가 높은 상위 5개 영화 추출

item_sim_df["(500) Days of Summer (2009)"].sort_values(ascending=False)[1:6]
# 'item_sim_df'는 영화 간 코사인 유사도 행렬로, 각 셀은 두 영화 간 유사도 점수를 의미함
# ["(500) Days of Summer (2009)"]는 해당 영화를 기준으로 다른 영화들과의 유사도 열을 선택함
# sort_values(ascending=False)는 유사도 점수가 높은 순으로 정렬함
# [1:6]은 자기 자신(유사도 1.0인 첫 번째 항목)을 제외하고, 유사도가 높은 상위 5개의 다른 영화를 선택함
# 결과적으로 '500일의 썸머'와 가장 유사한 영화 5편의 제목과 유사도 점수를 출력함


title
Scott Pilgrim vs. the World (2010)    0.502121
Up in the Air (2009)                  0.498354
Social Network, The (2010)            0.497004
Forgetting Sarah Marshall (2008)      0.472271
Shutter Island (2010)                 0.468202
Name: (500) Days of Summer (2009), dtype: float64

In [ ]:
# 고객 유사도 행렬 생성
user_sim = cosine_similarity(df_merge_pivot_null)
# 사용자-영화 평점 행렬(df_merge_pivot_null)을 입력으로 하여 사용자 간 코사인 유사도를 계산함
# 각 사용자(행)는 영화 평점 벡터로 표현되며, cosine_similarity는 두 사용자 간 선호 패턴의 방향 유사도를 수치화함
# 반환 결과는 사용자 간 유사도를 나타내는 2차원 대칭 행렬로, 각 셀 값은 두 사용자 간의 유사도임
# 값 범위는 [0, 1]이며, 1에 가까울수록 두 사용자가 매우 유사한 영화 취향을 가짐을 의미함

# 데이터 프레임 형태 변환
user_sim_df = pd.DataFrame(user_sim, index=df_merge_pivot_null.index,
                           columns=df_merge_pivot_null.index)
# 계산된 유사도 행렬(user_sim)을 pandas DataFrame으로 변환함
# index와 columns에 사용자 ID를 동일하게 지정하여 해석 가능한 유사도 행렬 구성함
# 결과적으로 각 행과 열은 사용자이며, 셀 값은 해당 두 사용자 간의 평점 기반 유사도임

user_sim_df.head()
# 생성된 사용자 간 유사도 행렬의 상위 5개 행을 출력함
# 각 행은 특정 사용자를 기준으로, 열에는 다른 사용자와의 유사도가 나열됨
# 대각선은 항상 1.0이며(자기 자신과의 유사도), 비대각 요소를 통해 취향이 유사한 사용자 탐색 가능함


userId,1,2,3,4,5,6,7,8,9,10,...,659,660,661,662,663,664,665,666,667,668
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.101113,0.210044,0.128766,0.057896,0.077130,0.358090,0.097434,0.239189,0.026663,...,0.291162,0.144741,0.106583,0.091049,0.236805,0.154519,0.245071,0.238660,0.278217,0.153493
2,0.101113,1.000000,0.115559,0.034610,0.032705,0.028305,0.062914,0.471918,0.194232,0.000000,...,0.068325,0.000000,0.477330,0.146887,0.163553,0.061737,0.050948,0.051423,0.035907,0.064822
3,0.210044,0.115559,1.000000,0.058208,0.044426,0.012816,0.084522,0.066620,0.459703,0.068454,...,0.152078,0.301021,0.081626,0.098949,0.310234,0.079452,0.092821,0.080940,0.158943,0.109658
4,0.128766,0.034610,0.058208,1.000000,0.019298,0.005781,0.059089,0.024420,0.050572,0.000000,...,0.055860,0.024329,0.040467,0.108881,0.076241,0.014011,0.042643,0.174275,0.061677,0.157809
5,0.057896,0.032705,0.044426,0.019298,1.000000,0.053378,0.080822,0.041536,0.023168,0.011915,...,0.058450,0.007315,0.024708,0.038163,0.053085,0.048993,0.055431,0.026053,0.086667,0.068281


In [ ]:
# 7번 고객과 유사도가 높은 상위 5명 추출

user_sim_df[7].sort_values(ascending=False)[1:6]
# user_sim_df는 사용자 간 코사인 유사도 행렬로, 각 셀은 두 사용자 간 유사도 점수를 의미함
# [7]은 7번 사용자(userId = 7)를 기준으로 다른 사용자들과의 유사도를 열 방향으로 선택함
# sort_values(ascending=False)는 유사도 값을 기준으로 내림차순 정렬함
# [1:6]은 유사도 1.0인 자기 자신(7번 사용자)을 제외하고, 유사도가 가장 높은 사용자 5명을 추출함
# 결과는 7번 사용자와 취향이 가장 비슷한 다른 사용자 5명의 ID 및 유사도 점수를 나타냄


userId
403    0.432287
358    0.414600
228    0.396949
328    0.391268
590    0.387817
Name: 7, dtype: float64

In [ ]:
# 협업 필터링용 샘플 행렬 생성

# 잠재요인 차원 30으로 설정
K = 30
# 행렬 분해(Matrix Factorization)에서 사용자와 아이템을 잠재요인(latent factors) 공간으로 임베딩하기 위한 차원 수를 설정함
# K는 사용자 선호와 아이템 특성을 추상적으로 표현할 수 있는 내재된 특성의 수를 의미함
# 일반적으로 K값이 너무 작으면 정보 손실이 발생하고, 너무 크면 과적합 위험이 있음

# 샘플용 영화 30개만 필터링
df_merge_sample = df_merge_pivot.iloc[:, 0:30]
# 전체 평점 행렬(df_merge_pivot)에서 앞쪽 30개 영화만 선택하여 샘플 데이터로 사용함
# 학습 속도 및 시각화를 고려하여 협업 필터링 실습용 소규모 행렬 구성 목적

df_array = df_merge_sample.values
# DataFrame을 NumPy 배열로 변환함
# 계산 효율성을 높이기 위해 Pandas 구조를 제거하고 수치 계산 가능한 형태로 구성함

user_cnt, item_cnt = df_array.shape
# 전체 사용자 수(user_cnt)와 아이템 수(item_cnt)를 배열의 행렬 크기에서 추출함
# 이후 사용자/아이템 행렬을 생성할 때 기준 차원으로 사용함

# 고객수, 영화 수 x 자원 수 행렬 생성
np.random.seed(47)
# 난수 고정(seed) 설정을 통해 동일한 난수 행렬을 재현 가능하도록 설정함
# 실험 결과 재현성(reproducibility)을 확보하기 위한 표준적인 처리 방식임

user_matrix = np.random.normal(scale=1./K, size=(user_cnt, K))
# 사용자 행렬을 무작위로 초기화함
# 각 사용자는 K차원 잠재요인 벡터를 가지며, 평균 0, 표준편차 1/K인 정규분포로 초기화함
# shape=(user_cnt, K)는 사용자 수 × 잠재요인 수로 구성됨

item_matrix = np.random.normal(scale=1./K, size=(item_cnt, K))
# 아이템 행렬도 동일하게 무작위로 초기화함
# 각 아이템은 K차원 잠재요인 벡터를 가지며, 정규분포 기반으로 생성됨
# shape=(item_cnt, K)는 영화 수 × 잠재요인 수로 구성됨

print("고객 행렬 확인:", user_matrix.shape)
# 생성된 사용자 행렬의 차원을 출력함
# 사용자 기반 임베딩 공간 구성이 잘 되었는지 확인하는 용도임

print("영화 행렬 확인:", item_matrix.shape)
# 생성된 아이템 행렬의 차원을 출력함
# 추후 내적(dot product)을 통해 평점 예측에 사용될 예정임


고객 행렬 확인: (668, 30)
영화 행렬 확인: (30, 30)


| 행렬            | 크기        | 의미               |
| ------------- | --------- | ---------------- |
| `user_matrix` | 사용자 수 × K | 각 사용자의 잠재적 선호 벡터 |
| `item_matrix` | 아이템 수 × K | 각 아이템의 잠재적 특성 벡터 |


In [ ]:
# RMSE 함수 정의

def get_rmse(df_array, user_matrix, item_matrix, not_nan_index):
    error = 0
    # 함수 호출 시 불필요한 초기화이며 실질적으로 사용되지 않음 (불필요한 변수 선언임)

    # 예측용 df_array 생성
    pred_rating_matrix = user_matrix @ item_matrix.T
    # 사용자 행렬(user_matrix)과 아이템 행렬(item_matrix)의 내적을 수행하여 예측 평점 행렬 생성
    # @ 연산자는 행렬 곱(dot product) 연산을 의미함
    # 각 원소는 사용자의 잠재요인 벡터와 아이템의 잠재요인 벡터 간 내적 결과로, 예측 평점에 해당함

    # 결측 없는 실제 행렬과 예측 행렬 생성
    df_array_not_null = df_array[not_nan_index]
    # 실제 평점 행렬(df_array)에서 결측이 없는 인덱스(관측값 위치)만 필터링하여 정답 벡터 구성
    # not_nan_index는 평점이 존재하는 위치의 좌표 정보 배열임 (예: np.where(df_array > 0))

    pred_rating_matrix_not_null = pred_rating_matrix[not_nan_index]
    # 예측된 평점 행렬에서 동일한 위치(관측값이 있던 곳)만 추출하여 예측 벡터 구성
    # 이로써 실제 평점과 예측 평점이 1:1로 비교 가능한 상태로 정렬됨

    # RMSE 산출
    mse = mean_squared_error(df_array_not_null, pred_rating_matrix_not_null)
    # 실제 평점과 예측 평점 간 평균 제곱 오차(MSE)를 계산함
    # sklearn의 mean_squared_error 함수는 두 벡터 간 차이를 제곱하고 평균을 구함

    rmse = np.sqrt(mse)
    # MSE에 루트를 씌워 RMSE(Root Mean Squared Error)로 변환함
    # RMSE는 예측 성능을 평가하는 대표적인 지표로, 값이 작을수록 예측 정확도가 높음을 의미함

    return rmse
    # 최종적으로 계산된 RMSE 값을 반환함


| 매개변수            | 설명                                               |
| --------------- | ------------------------------------------------ |
| `df_array`      | 실제 평점 배열 (사용자 × 아이템)                             |
| `user_matrix`   | 사용자 잠재요인 행렬 (U × K)                              |
| `item_matrix`   | 아이템 잠재요인 행렬 (I × K)                              |
| `not_nan_index` | 실제 평점이 존재하는 위치 인덱스 (예: `np.where(df_array > 0)`) |


### RMSE란?

- **Root Mean Squared Error (RMSE)**
    
- 예측된 평점과 실제 평점 간의 평균적인 차이를 **루트 제곱**으로 표현한 지표
- 값이 **작을수록 예측 정확도**가 높음을 의미함
- 단위는 **평점 단위**와 동일함 (예: 0.5 ~ 5.0 사이)

In [ ]:
# 행렬 분해 함수 정의

def matrix_factorization(df_array, K, steps=1000, 
                         learning_rate=0.01, r_lambda=0.01):
    # 사용자-아이템 평점 행렬(df_array)을 K차원 잠재요인으로 분해하는 함수 정의
    # 확률적 경사 하강법(SGD)을 사용하여 반복적으로 사용자/아이템 행렬을 갱신함
    # steps: 최대 반복 횟수, learning_rate: 학습률, r_lambda: 정규화 계수

    # 결측값이 아닌 df_array의 index 생성
    not_nan_index = np.where(np.isnan(df_array) == False)
    # 실제 평점이 존재하는 위치의 (행, 열) 인덱스를 튜플 형태로 추출함
    # 행렬 분해는 관측된 평점만을 대상으로 학습해야 하므로 이 인덱스를 기준으로 학습을 진행함

    # SGD 행렬 분해 알고리즘 적용
    for step in range(steps):
        # 전체 반복 횟수만큼 학습을 수행함 (기본값 1000회)

        for p, q, r in zip(not_nan_index[0], not_nan_index[1], df_array[not_nan_index]):
            # 관측된 평점 하나에 대해 반복 수행함
            # p: 사용자 인덱스, q: 아이템 인덱스, r: 실제 평점

            # 실제 값과 예측 값 차이 계산
            r_pq = user_matrix[p, :] @ item_matrix[q, :].T
            # 현재 사용자-아이템 쌍에 대한 내적 결과를 예측 평점으로 계산함
            error_pq = r - r_pq
            # 예측값과 실제값 간 오차를 계산함

            # SGD - 사용자 행렬 업데이트
            user_matrix[p, :] = user_matrix[p, :] + learning_rate * (
                error_pq * item_matrix[q, :] - r_lambda * user_matrix[p, :])
            # 예측 오차를 반영하여 사용자 벡터를 갱신함
            # 정규화 항(r_lambda)은 과적합을 방지하기 위한 L2 패널티 역할을 수행함

            # SGD - 아이템 행렬 업데이트
            item_matrix[q, :] = item_matrix[q, :] + learning_rate * (
                error_pq * user_matrix[p, :] - r_lambda * item_matrix[q, :])
            # 동일한 방식으로 아이템 벡터도 예측 오차를 반영하여 갱신함

        rmse = get_rmse(df_array, user_matrix, item_matrix, not_nan_index)
        # 현재 step에서 사용자/아이템 행렬을 기반으로 전체 예측 행렬을 계산하고 RMSE를 측정함
        # 성능 추이를 확인하기 위한 모니터링 용도로 사용함

        if ( (step + 1) % 100) == 0:
            print("반복 횟수: ", step + 1, " RMSE: ", np.round(rmse, 3))
            # 매 100회 학습마다 현재 RMSE를 출력하여 학습 경과를 추적함

    return user_matrix, item_matrix
    # 학습 완료된 사용자 행렬과 아이템 행렬을 반환함


| 구성 요소                        | 설명                                    |
| ---------------------------- | ------------------------------------- |
| `not_nan_index`              | 실제 평점이 존재하는 위치만 대상으로 학습 수행            |
| `user_matrix`, `item_matrix` | K차원 잠재요인 벡터로 초기화된 사용자 및 아이템 행렬        |
| `SGD`                        | 관측된 평점에 대해 예측 오차를 기반으로 두 행렬을 반복적으로 갱신 |
| `RMSE`                       | 반복 중간에 예측 성능을 점검하기 위한 지표로 사용          |
| `return`                     | 학습 완료된 잠재요인 행렬 반환 (추천 시스템에 활용 가능)     |


In [11]:
# 행렬 분해, 내적

user_matrix, item_matrix = matrix_factorization(
    df_array, K, steps=1000,
    learning_rate=0.01, r_lambda = 0.01)

pred_matrix = user_matrix @ item_matrix.T

반복 횟수:  100  RMSE:  0.097
반복 횟수:  200  RMSE:  0.027
반복 횟수:  300  RMSE:  0.024
반복 횟수:  400  RMSE:  0.023
반복 횟수:  500  RMSE:  0.021
반복 횟수:  600  RMSE:  0.02
반복 횟수:  700  RMSE:  0.02
반복 횟수:  800  RMSE:  0.019
반복 횟수:  900  RMSE:  0.019
반복 횟수:  1000  RMSE:  0.018


### 실행 결과 구조

| 행렬 이름 | 크기 | 내용 |
| --- | --- | --- |
| `user_matrix` | 사용자 수 × K | 각 사용자의 잠재 요인 벡터 |
| `item_matrix` | 아이템 수 × K | 각 아이템의 잠재 요인 벡터 |
| `pred_matrix` | 사용자 수 × 아이템 수 | 각 사용자-아이템 조합에 대한 **예측 평점** |

In [ ]:
# 데이터 프레임 변환
ratings_pred_matrix = pd.DataFrame(data=pred_matrix, 
                                   index=df_merge_sample.index,
                                   columns=df_merge_sample.columns)
# NumPy 배열(pred_matrix)을 pandas DataFrame으로 변환함
# 행 인덱스(index)는 실제 사용자 ID(df_merge_sample.index)를 사용하여 원래 사용자 정보를 보존함
# 열 이름(columns)은 영화 제목(df_merge_sample.columns)으로 설정하여, 해석 가능한 구조로 만듦
# 결과적으로 사용자-영화 기반 예측 평점 테이블을 생성함
# 각 셀의 값은 학습된 행렬 분해 모델로부터 계산된 **예측 평점**을 의미함

ratings_pred_matrix.head(5)
# 예측 평점 행렬의 상위 5개 행을 출력함
# 각 행은 특정 사용자, 각 열은 영화이며, 셀 값은 예측 평점으로 구성됨


title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Til There Was You (1997),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...And Justice for All (1979),10 (1979),...,"10th Kingdom, The (2000)",11-11-11 (11-11-11: The Prophecy) (2011),11:14 (2003),"11th Hour, The (2007)",12 (2007),12 Angry Men (1957),12 Angry Men (1997),12 Rounds (2009),12 Years a Slave (2013),127 Hours (2010)
userId,,,,,,,,,,,,,,,,,,,,,
1,-0.051631,-0.083938,-0.069660,-0.105834,-0.186691,-0.094306,-0.098581,-0.177139,-0.103241,-0.082794,...,-0.045882,-0.079783,-0.112016,-0.110756,-0.034177,-0.163101,-0.076059,-0.051933,-0.128448,-0.113047
2,0.068995,0.063763,0.029378,0.103909,0.070665,0.035862,0.094289,0.170707,0.130561,0.053904,...,0.134833,0.066054,0.067439,0.097416,0.013622,0.127945,-0.041964,0.049973,0.106233,0.082817
3,0.128438,0.089740,0.090381,0.066533,0.103108,0.031881,0.212484,0.048289,0.182902,0.070970,...,0.142718,0.191402,0.256842,0.230997,0.064861,0.149501,-0.003335,0.198401,0.207483,0.198376
4,2.895991,1.712282,1.787002,2.657818,2.820935,2.213617,2.082815,2.555981,3.306608,2.252366,...,1.648483,1.936613,2.729978,2.677794,1.100380,4.981980,0.620007,2.018014,3.723871,2.951902
5,0.001853,0.076697,-0.039216,-0.029270,-0.007633,-0.051874,-0.026496,0.004058,0.040571,-0.024389,...,0.042998,0.052845,0.024583,0.030629,-0.026081,-0.008237,0.040031,0.000849,0.007089,-0.021943


In [ ]:
# 영화 추천을 위한 함수 설정

# 미상영 영화 리스트 추출 함수
def get_unseen_movies(df_merge_sample, userId):
    # 특정 사용자가 아직 평가하지 않은(보지 않은) 영화 목록을 추출하는 함수 정의

    # 모든 영화 리스트 생성
    movies_list = df_merge_sample.columns.tolist()
    # 사용자-영화 평점 행렬의 열 이름을 리스트로 변환하여 전체 영화 목록을 생성함

    # 고객 별 평점 테이블 생성
    ratings = df_merge_sample.loc[userId, :]
    # 특정 사용자의 모든 영화에 대한 평점 정보를 추출함 (Series 형태)

    # 평점을 메기지 않은 영화 리스트 생성
    none_rating_list = ratings[ratings.isnull()].index.tolist()
    # 평점이 NaN인 영화의 인덱스를 추출하여, 해당 사용자가 아직 보지 않은 영화 목록으로 구성함

    # 평점 없는 영화로 미상영 영화 리스트 생성
    unseen_movie_list = [movie for movie in movies_list if movie in none_rating_list]
    # 전체 영화 중 사용자가 보지 않은 영화만 필터링하여 최종 미관람 영화 리스트를 생성함

    return unseen_movie_list
    # 해당 사용자가 아직 보지 않은 영화 제목 리스트를 반환함

# 미상영 영화 중 예측 점수가 높은 순으로 정렬
def recomm_movie_by_userid(pred_df, userId, unseen_movie_list, top_n=10):
    # 특정 사용자(userId)에 대해 보지 않은 영화들 중 예측 평점이 높은 순으로 top-N 추천하는 함수 정의

    recomm_movies = pred_df.loc[userId, unseen_movie_list].sort_values(ascending=False)[:top_n]
    # 예측 평점 데이터프레임(pred_df)에서 해당 사용자의 미관람 영화 리스트를 기준으로 예측 평점을 추출함
    # 내림차순 정렬 후 상위 N개 영화만 슬라이싱하여 추천 리스트로 구성함

    return recomm_movies
    # 추천 영화 리스트를 시리즈 형태로 반환함 (index: 영화 제목, value: 예측 평점)

In [ ]:
# 575번 고객의 추천 영화 리스트 생성

# 575번 고객의 미상영 영화 리스트 생성
unseen_movie_list = get_unseen_movies(df_merge_sample, 575)
# 사용자 ID가 575인 고객이 아직 시청(평점 부여)하지 않은 영화 리스트를 추출함
# df_merge_sample은 실제 평점 기반의 사용자-영화 행렬이며, NaN은 미관람 상태를 의미함

# 미상영 영화 중 예측 평점 높은 영화 리스트 생성
recomm_movies = recomm_movie_by_userid(ratings_pred_matrix, 575,
                                       unseen_movie_list, top_n=10)
# 예측 평점 행렬(ratings_pred_matrix)에서 575번 사용자의 미관람 영화 중
# 예측 평점이 높은 상위 10개 영화를 추출함
# recomm_movies는 (index: 영화 제목, value: 예측 평점)의 Series 형태임

# 최종 데이터셋 생성
recomm_movies = pd.DataFrame(data=recomm_movies.values,
                             index=recomm_movies.index,
                             columns=['pred_score']).reset_index()
# 추천 영화 리스트를 pandas DataFrame으로 변환함
# 영화 제목을 인덱스에서 일반 컬럼으로 이동하고 컬럼 이름을 'pred_score'로 설정함
# reset_index()를 사용하여 시각적으로 보기 좋은 테이블 형태로 정비함

recomm_movies.head(10)
# 575번 고객에게 추천된 영화 리스트 상위 10개를 출력함
# 각 행은 영화 제목과 예측 평점으로 구성되며, 평점이 높은 순서대로 정렬됨


,title,pred_score
0,12 Years a Slave (2013),3.569347
1,127 Hours (2010),3.362532
2,101 Dalmatians (One Hundred and One Dalmatians...,2.968062
3,10 Items or Less (2006),2.949394
4,11:14 (2003),2.884700
5,"11th Hour, The (2007)",2.742791
6,*batteries not included (1987),2.701928
7,'71 (2014),2.672483
8,12 Rounds (2009),2.585290
9,10th & Wolf (2006),2.489459


### 전체 로직 흐름 요약

1. `get_unseen_movies()`
    
    → 사용자가 보지 않은 영화 찾기
    
2. `recomm_movie_by_userid()`
    
    → 예측 평점 기반 추천 리스트 생성
    
3. `DataFrame 변환`
    
    → 출력 및 후처리를 위한 정돈된 추천 테이블 완성